In [39]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import pandas as pd
import os
import plotly.express as px
load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

df_conn_test = pd.read_sql("SELECT * FROM abfragen LIMIT 5", engine)
df_conn_test.head()

,id,tankstellen_id,diesel,e5,e10,isopen,retrieval_time,retrieval_date
0,97,a7cdd9cf-b467-4aac-8eab-d662f082511e,1.689,1.759,1.699,True,11:47,2026-02-15
1,98,e102dc2a-a9db-4a70-adfa-4abd42ad7dcb,1.699,1.759,1.699,True,11:47,2026-02-15
2,99,4f4eadab-2845-4504-8acc-50a19bec8583,1.699,1.759,1.699,True,11:47,2026-02-15
3,100,4d726ab3-6ad3-45c1-a652-362ae6d62c6b,1.699,1.759,1.699,True,11:47,2026-02-15
4,101,6903db69-dd2c-4a8c-b657-caec419c7a32,1.699,1.759,1.699,True,11:47,2026-02-15


In [40]:
df = pd.read_sql("SELECT * FROM abfragen", engine)

# Grundüberblick
print(f"Gesamte Datensätze: {len(df)}")
print(f"Zeitraum: {df['retrieval_date'].min()} bis {df['retrieval_date'].max()}")
print(f"Anzahl eindeutige Tankstellen: {df['tankstellen_id'].nunique()}")
print(f"Anzahl eindeutige Tage: {df['retrieval_date'].nunique()}")

# Datensätze pro Tag
print(f"\nDatensätze pro Tag:")
print(df.groupby('retrieval_date').size())

Gesamte Datensätze: 188640
Zeitraum: 2026-02-15 bis 2026-03-28
Anzahl eindeutige Tankstellen: 96
Anzahl eindeutige Tage: 42

Datensätze pro Tag:
retrieval_date
2026-02-15    2496
2026-02-16    4608
2026-02-17    4608
2026-02-18    4608
2026-02-19    4608
2026-02-20    4608
2026-02-21    4608
2026-02-22    4608
2026-02-23    4608
2026-02-24    4608
2026-02-25    4608
2026-02-26    4608
2026-02-27    4608
2026-02-28    4608
2026-03-01    4608
2026-03-02    4608
2026-03-03    4608
2026-03-04    4608
2026-03-05    4608
2026-03-06    4608
2026-03-07    4608
2026-03-08    4608
2026-03-09    4608
2026-03-10    4608
2026-03-11    4608
2026-03-12    4608
2026-03-13    4608
2026-03-14    4608
2026-03-15    4608
2026-03-16    4608
2026-03-17    4608
2026-03-18    4608
2026-03-19    4608
2026-03-20    4608
2026-03-21    4608
2026-03-22    4608
2026-03-23    4608
2026-03-24    4608
2026-03-25    4608
2026-03-26    4608
2026-03-27    4608
2026-03-28    1824
dtype: int64


In [41]:
# Fehlende Werte
print("Fehlende Werte pro Spalte:")
print(df.isnull().sum())

print(f"\nDatentypen:")
print(df.dtypes)

Fehlende Werte pro Spalte:
id                  0
tankstellen_id      0
diesel             62
e5                302
e10                22
isopen              0
retrieval_time      0
retrieval_date      0
dtype: int64

Datentypen:
id                  int64
tankstellen_id        str
diesel            float64
e5                float64
e10               float64
isopen               bool
retrieval_time        str
retrieval_date     object
dtype: object


In [42]:
df_stations = pd.read_sql("SELECT * FROM tankstellen", engine)
df_stations.head()

,id,name,brand,street,place,lat,lng,dist,housenumber,postcode
0,a7cdd9cf-b467-4aac-8eab-d662f082511e,STUTTGART - KRIEGSBERGSTRASSE 55 A,AGIP ENI,Kriegsbergstrasse,Stuttgart,48.782290,9.171490,0.8,55 A,70174
1,e102dc2a-a9db-4a70-adfa-4abd42ad7dcb,AVIA S-Bebelstr.,AVIA,Bebelstr.,Stuttgart,48.774800,9.158100,1.0,9,70176
2,4f4eadab-2845-4504-8acc-50a19bec8583,Shell Stuttgart Rosenbergplatz 7,Shell,Rosenbergplatz,Stuttgart,48.778509,9.155544,1.3,7,70193
3,4d726ab3-6ad3-45c1-a652-362ae6d62c6b,Esso Station,ESSO,Immenhofer Str. 48,Stuttgart,48.762952,9.176572,1.4,,70180
4,6903db69-dd2c-4a8c-b657-caec419c7a32,Shell Stuttgart Karl-Kloss-Str. 18,Shell,Karl-Kloss-Str.,Stuttgart,48.759586,9.159391,1.9,18,70199


In [43]:
fig = px.scatter_map(
    df_stations,
    lat="lat",
    lon="lng",
    hover_name="name",
    hover_data=["brand"],
    zoom=10,
    map_style="open-street-map",
)


fig.update_traces(marker=dict(size=15, color="blue", opacity=0.5))
fig.show()
# Karte als HTML speichern (Karte geht nicht als PNG)
fig.write_html("assets/karte.html")


In [44]:
# Durchschnittspreis pro Messpunkt (retrieval_date + retrieval_time)
df_avg = df.groupby('retrieval_date')[['diesel', 'e5', 'e10']].mean().reset_index()

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(x=df_avg['retrieval_date'], y=df_avg['diesel'], name='Diesel', mode='lines'))
fig.add_trace(go.Scatter(x=df_avg['retrieval_date'], y=df_avg['e5'], name='E5', mode='lines'))
fig.add_trace(go.Scatter(x=df_avg['retrieval_date'], y=df_avg['e10'], name='E10', mode='lines'))

fig.update_layout(
    title='Durchschnittlicher Spritpreis im Raum Stuttgart',
    xaxis_title='Datum',
    yaxis_title='Preis (€)',
    hovermode='x unified'
)

fig.update_xaxes(
    dtick="D1",
    tickformat="%d.%m",
    tickangle=45
)


fig.add_vline(
    x=pd.Timestamp("2026-02-28").timestamp() * 1000,
    line_dash="dash",
    line_color="red",
    annotation_text="28.02.",
    annotation_position="top"
)

fig.show()

# Preisverlauf speichern
fig.write_image("assets/preisverlauf.png", width=1668, height=450)


In [46]:
brand_counts = df_stations['brand'].value_counts().reset_index()
brand_counts.columns = ['brand', 'anzahl']
print(brand_counts)

fig = px.bar(
    brand_counts,
    x='brand',
    y='anzahl',
    title='Anzahl Tankstellen pro Marke im Raum Stuttgart',
    labels={'brand': 'Marke', 'anzahl': 'Anzahl'},
    text='anzahl'
)

fig.update_layout(xaxis_tickangle=45)
fig.show()
fig.write_image("assets/brand_counts.png", width=1668, height=450)

                               brand  anzahl
0                              Shell      18
1                               ARAL      16
2                               ESSO      12
3                           AGIP ENI       9
4                                JET       9
5                               AVIA       8
6                        AVIA XPress       3
7                      TotalEnergies       3
8                                RAN       3
9              Supermarkt-Tankstelle       3
10                                SB       1
11                  Freie Tankstelle       1
12           Mr. Wash Autoservice AG       1
13                            Sprint       1
14                            Access       1
15                 SCHARR-Tankstelle       1
16                     ORLEN Express       1
17                               BFT       1
18                               HEM       1
19                     SB Tankstelle       1
20                  freie Tankstelle       1
21  MTB Au